# OceanEmbed — Output Exploration

Sanity-checks for the pipeline output, meant to run **after** `load_data.py` and `split_data.py` have produced files in `data/processed/`. Not part of the pipeline itself — this is the debugging/exploration companion to the scripts in the repo root.

Sections:
1. Load merged inputs/target, check shapes and NaN coverage
2. Confirm split day counts match what `split_data.py` printed
3. Plot seasonal-quarter coverage per split (the thing the whole seasonal-split design is meant to guarantee)
4. Spot-check one regridded variable on a map

In [ ]:
import sys
sys.path.insert(0, "..")  # so `from config import PROCESSED_DIR` resolves from repo root

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import PROCESSED_DIR, LON_MIN, LON_MAX, LAT_MIN, LAT_MAX

PROCESSED_DIR

## 1. Load merged inputs/target — shape and NaN sanity check

These are the pre-split files written by `load_data.py`. Confirm dims line up and NaN coverage isn't unexpectedly high (a regridding or alignment bug usually shows up here first).

In [ ]:
inputs = xr.open_dataset(PROCESSED_DIR / "surface_inputs.nc")
target = xr.open_dataset(PROCESSED_DIR / "glorys_target.nc")

print("surface_inputs.nc")
print(inputs)
print("\nglorys_target.nc")
print(target)

In [ ]:
# NaN fraction per variable — flag anything above a sanity threshold
def nan_report(ds, label):
    print(f"--- {label} ---")
    for var in ds.data_vars:
        frac = float(ds[var].isnull().mean())
        flag = "  <-- check this" if frac > 0.5 else ""
        print(f"{var:20s} {frac:6.1%} NaN{flag}")

nan_report(inputs, "surface_inputs")
print()
nan_report(target, "glorys_target")

In [ ]:
# Time axis alignment — inputs and target should share the exact same days
inputs_times = pd.DatetimeIndex(inputs.time.values)
target_times = pd.DatetimeIndex(target.time.values)

print(f"inputs:  {len(inputs_times)} days, {inputs_times.min().date()} to {inputs_times.max().date()}")
print(f"target:  {len(target_times)} days, {target_times.min().date()} to {target_times.max().date()}")
print(f"mismatch: {len(inputs_times.symmetric_difference(target_times))} days not shared")

## 2. Confirm split day counts

`split_data.py` prints per-quarter, per-split day counts as it runs. Cross-check those numbers against what actually landed in the six split files.

In [ ]:
split_names = ["train", "val", "test"]

split_days = {}
for split in split_names:
    ds = xr.open_dataset(PROCESSED_DIR / f"surface_inputs_{split}.nc")
    split_days[split] = pd.DatetimeIndex(ds.time.values)
    ds.close()

total = sum(len(v) for v in split_days.values())
for split, times in split_days.items():
    print(f"{split:6s} {len(times):4d} days  ({len(times)/total:5.1%})")
print(f"{'total':6s} {total:4d} days")

In [ ]:
# No day should appear in more than one split — this must be empty
train_val_overlap = split_days["train"].intersection(split_days["val"])
val_test_overlap = split_days["val"].intersection(split_days["test"])
train_test_overlap = split_days["train"].intersection(split_days["test"])

print(f"train/val overlap:  {len(train_val_overlap)} days")
print(f"val/test overlap:   {len(val_test_overlap)} days")
print(f"train/test overlap: {len(train_test_overlap)} days")
assert len(train_val_overlap) == len(val_test_overlap) == len(train_test_overlap) == 0, "Splits overlap — investigate split_data.py"

## 3. Seasonal-quarter coverage per split

The whole point of the seasonal-quarter split (vs. a naive Jan–Aug/Sep–Oct/Nov–Dec cut) is that every split sees every season. This plot is the direct check of that: each split's days should spread across all four quarters, not cluster in one.

In [ ]:
from split_data import QUARTERS

def quarter_of(ts, quarters=QUARTERS):
    for label, start, end in quarters:
        if pd.Timestamp(start) <= ts <= pd.Timestamp(end):
            return label
    return "unassigned"

rows = []
for split, times in split_days.items():
    for t in times:
        rows.append({"split": split, "quarter": quarter_of(t), "date": t})

coverage = pd.DataFrame(rows)
pivot = coverage.pivot_table(index="quarter", columns="split", values="date", aggfunc="count", fill_value=0)
pivot = pivot.reindex([q[0] for q in QUARTERS])  # keep chronological quarter order
pivot

In [ ]:
pivot[split_names].plot(kind="bar", figsize=(8, 4))
plt.title("Days per split, per seasonal quarter")
plt.ylabel("days")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# Every quarter should have a non-zero bar in every split -- a zero here means
# that split never sees that season, which defeats the purpose of the design.
missing = pivot[(pivot[split_names] == 0).any(axis=1)]
if len(missing):
    print("WARNING — these quarters are missing from at least one split:")
    print(missing)
else:
    print("OK — every split has coverage in every quarter.")

In [ ]:
# Timeline view: each split's days across the full year, colored by split.
# Purge gaps should show up as small blank strips at quarter-internal boundaries.
fig, ax = plt.subplots(figsize=(12, 2))
colors = {"train": "tab:blue", "val": "tab:orange", "test": "tab:green"}

for split, times in split_days.items():
    ax.scatter(times, [split] * len(times), s=4, color=colors[split], label=split)

for label, start, end in QUARTERS:
    ax.axvline(pd.Timestamp(start), color="gray", linestyle="--", linewidth=0.5)

ax.set_title("Split assignment across the year (dashed lines = quarter boundaries)")
plt.tight_layout()
plt.show()

## 4. Spot-check a regridded variable on a map

Quick visual check that the 0.25° regrid landed correctly over the study region and isn't, e.g., flipped, shifted, or full of edge artifacts. Swap `VAR_TO_CHECK` for whatever's actually in your merged `surface_inputs.nc` (variable names depend on the upstream product's naming).

In [ ]:
VAR_TO_CHECK = list(inputs.data_vars)[0]  # replace with e.g. "analysed_sst" once you know the real name
DAY_TO_CHECK = inputs.time.values[0]

snapshot = inputs[VAR_TO_CHECK].sel(time=DAY_TO_CHECK)

fig, ax = plt.subplots(figsize=(7, 5))
snapshot.plot(ax=ax, x="lon", y="lat")
ax.set_xlim(LON_MIN, LON_MAX)
ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_title(f"{VAR_TO_CHECK} — {pd.Timestamp(DAY_TO_CHECK).date()}")
plt.tight_layout()
plt.show()

In [ ]:
inputs.close()
target.close()